In [196]:
from keras import layers
from keras import Input
from keras.models import Model

import numpy as np

import keras
import tensorflow as tf
import os
import csv
import pathlib
import unicode

import matplotlib as plt

#from tensorflow.python.keras.preprocessing.image import ImageDataGenerator

In [235]:
def get_dataset_fromCsv(path):
    cnt = 0
    print("Helelo?")
    
    csvFile = open(path, 'r', encoding='utf-8')
    reader = csv.reader(csvFile)
    
    for line in reader:
        if cnt > 100:
            break
        else :
            imgFile = line[0]
            
            def datasetGenerator():
                #img = np.random.rand(64,64,3)
                img = tf.io.read_file(imgFile)
                img = tf.image.decode_jpeg(img, channels=3)
                img = tf.image.convert_image_dtype(img, tf.float32)
                img = tf.image.resize(img, (64, 64))    
                
                sent1 = np.array([1, 2, 3, 4, 5, 6, 7, 8], dtype=np.int64)
                sent1 = np.reshape(sent1, (8, 1, 1))
                
                print("datasetGenerator")
                #yield img, {"a" : 11, "b" : 22, "c" : 33}
                yield img, {"a" : sent1, "b" : sent1, "c" : sent1}
                
        dataset = tf.data.Dataset.from_generator(datasetGenerator,
                                            output_types= (tf.float32, {"a" : tf.int16, "b" : tf.int16, "c" : tf.int16})
                                            )
        
        # dataset = tf.data.Dataset.from_generator(datasetGenerator,
        #                             output_types= (tf.float32, {"a" : tf.int16, "b" : tf.int16, "c" : tf.int16}),
        #                             #output_shapes=((64,64,3), {"a" : tf.TensorShape(19), "b" : tf.TensorShape(21), "c" : tf.TensorShape(28)})
        #                             output_shapes = None
        #                             )


        # dataset = tf.data.Dataset.from_generator(datasetGenerator,
        #                                          output_types= (tf.float32, {"a" : tf.int16, "b" : tf.int16, "c" : tf.int16}),
        #                                          output_shapes=((64,64,3), {"a" : tf.TensorShape(19), "b": tf.TensorShape(21), "c": tf.TensorShape(28)})                                                 
        #                                          )
                
        
        # dataset = tf.data.Dataset.from_generator(datasetGenerator,
        #                                          output_types= (tf.float32, {"a" : tf.int16, "b" : tf.int16, "c" : tf.int16}),
        #                                          output_shapes=({"posts": tf.TensorShape([None, 64,64,3])},
        #                                                 {"a": tf.TensorShape([19]), "b": tf.TensorShape([21]), "c": tf.TensorShape([28])})                                              
        #                                          )
        
#         dataset = tf.data.Dataset.from_generator(datasetGenerator,
#                                                  output_signature=
#                                                  (tf.TensorSpec(shape = (None, 64, 64, 3), dtype =tf.float32, name = "posts"),
                                                  
#                                                  tf.TensorSpec(shape = (3,), dtype = tf.int64, name = "DenseCho2")
                                                  
#                                                   )
#                                                  )

        #dataset = dataset.batch(2)
        return dataset
# dataset = tf.data.Dataset.from_generator(generator, output_types=({"input_1": tf.int64, "input_2": tf.int64}, tf.int64))
#model.fit({'a_input': x_train, 'b_input': x_train}, y_train, validation_data=({'a_input': x_valid, 'b_input': x_valid}, y_valid), epochs=10)            

In [236]:
dataset =  get_dataset_fromCsv("/root/Data/hangul/dataset/tranDataset.csv")

print(dataset)
iterator = iter(dataset)
print(next(iterator))


Helelo?
<_FlatMapDataset element_spec=(TensorSpec(shape=<unknown>, dtype=tf.float32, name=None), {'a': TensorSpec(shape=<unknown>, dtype=tf.int16, name=None), 'b': TensorSpec(shape=<unknown>, dtype=tf.int16, name=None), 'c': TensorSpec(shape=<unknown>, dtype=tf.int16, name=None)})>
datasetGenerator
(<tf.Tensor: shape=(64, 64, 3), dtype=float32, numpy=
array([[[1.       , 1.       , 1.       ],
        [1.       , 1.       , 1.       ],
        [1.       , 1.       , 1.       ],
        ...,
        [0.9960785, 0.9960785, 0.9960785],
        [0.9960785, 0.9960785, 0.9960785],
        [0.9960785, 0.9960785, 0.9960785]],

       [[1.       , 1.       , 1.       ],
        [1.       , 1.       , 1.       ],
        [1.       , 1.       , 1.       ],
        ...,
        [0.9960785, 0.9960785, 0.9960785],
        [0.9960785, 0.9960785, 0.9960785],
        [0.9960785, 0.9960785, 0.9960785]],

       [[1.       , 1.       , 1.       ],
        [1.       , 1.       , 1.       ],
        [1.   

In [237]:

posts_input = Input(shape=(64,64,3), dtype='float32', name='posts')

x = layers.Flatten()(posts_input)

DenseCho = layers.Dense(128, activation='relu', name='DenseCho1')(x)
DenseJung = layers.Dense(128, activation='relu', name='DenseJung1')(x)
DenseJong = layers.Dense(128, activation='relu', name='DenseJong1')(x)


DenseCho = layers.Dense(19, activation='softmax', name='a')(DenseCho)
DenseJung = layers.Dense(21, activation='softmax', name='b')(DenseJung)
DenseJong = layers.Dense(28, activation='softmax', name='c')(DenseJong)

losses = {
	#"DenseCho2": "categorical_crossentropy",
	"DenseCho2": "sparse_categorical_crossentropy",
	"DenseJung2": "sparse_categorical_crossentropy",
    "DenseJong2": "sparse_categorical_crossentropy"
}

model = Model(posts_input, [DenseCho, DenseJung, DenseJong])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=[['accuracy'], ['accuracy'], ['accuracy']])

# model.compile(loss=losses, optimizer='adam', metrics=[['accuracy'], ['accuracy'], ['accuracy']])
# model.compile(loss=["sparse_categorical_crossentropy", "sparse_categorical_crossentropy", "sparse_categorical_crossentropy"], 
#               optimizer='adam', 
#               metrics=[['accuracy'], ['accuracy'], ['accuracy']])


# model.compile(loss=["categorical_crossentropy", "categorical_crossentropy", "categorical_crossentropy"], 
#               optimizer='adam', 
#               metrics=[['accuracy'], ['accuracy'], ['accuracy']])

# model.compile(loss = 'sparse_categorical_crossentropy',optimizer='adam', 
#                metrics=[['accuracy'], ['accuracy'], ['accuracy']])

# model.compile(loss=["categorical_crossentropy", "categorical_crossentropy", "categorical_crossentropy"], 
#               optimizer='adam', 
#               metrics=[['accuracy'], ['accuracy'], ['accuracy']])

In [238]:
model.fit(get_dataset_fromCsv("/root/Data/hangul/dataset/tranDataset.csv"), epochs = 100, batch_size = 10)

Helelo?
Epoch 1/100


ValueError: as_list() is not defined on an unknown TensorShape.

In [ ]:
def get_dataset_fromCsv(path):
    cnt = 0
    print("Helelo?")
    
    csvFile = open(path, 'r', encoding='utf-8')
    reader = csv.reader(csvFile)
    
    for line in reader:
        if cnt > 100:
            break
        else :
            imgFile = line[0]
            
            img = tf.io.read_file(imgFile)
            img = tf.image.decode_jpeg(img, channels=3)
            img = tf.image.convert_image_dtype(img, tf.float32)
            img = tf.image.resize(img, (64, 64))    
            print(img.shape)
            print(imgFile)
            # print(label)

            cnt = cnt + 1
            #yield(img, label)
            def dataset_generator():
                labels = np.array([int(line[1]), int(line[2]), int(line[3])], dtype= 'int64')

                yield img, {"DenseCho2" : (int(line[1])), "DenseJung2" : (int(line[2])), "DenseJong2" : (int(line[3]))}


            
            
            dataset = tf.data.Dataset.from_generator(dataset_generator, 
                                         output_types=({"posts": tf.float32, },
                                                       {"DenseCho2": tf.int64, "DenseJung2": tf.int64, "DenseJong2": tf.int64}),
                                         #output_shapes=({"posts": tf.TensorShape([1, 64,64,3])},
                                         output_shapes=({"posts": tf.TensorShape([None, 64,64,3])},
                                                        {"DenseCho2": tf.TensorShape([19]), "DenseJung2": tf.TensorShape([21]), "DenseJong2": tf.TensorShape([28])}))

            return dataset
        
        
    
#get_dataset_fromCsv("/root/Data/hangul/dataset/tranDataset.csv")

In [5]:
def get_dataset_fromCsv(path):
    cnt = 0
    print("Helelo?")
    
    csvFile = open(path, 'r', encoding='utf-8')
    reader = csv.reader(csvFile)
    
    for line in reader:
        if cnt > 1:
            break
        else :
            imgFile = line[0]
            parse_image(imgFile)
            show(imgFile, line[1])


def parse_image(filename):
  parts = tf.strings.split(filename, os.sep)
  label = parts[-2]

  image = tf.io.read_file(filename)
  image = tf.io.decode_jpeg(image)
  image = tf.image.convert_image_dtype(image, tf.float32)
  image = tf.image.resize(image, [128, 128])
  return image, label

def show(image, label):
  plt.figure()
  plt.imshow(image)
  plt.title(label.numpy().decode('utf-8'))
  plt.axis('off')


  
get_dataset_fromCsv("/root/Data/hangul/dataset/tranDataset.csv")

Helelo?


2024-03-18 10:35:10.647568: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-18 10:35:10.666627: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-18 10:35:10.666666: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-18 10:35:10.668282: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-18 10:35:10.668326: I external/local_xla/xla/stream_executor

AttributeError: module 'matplotlib' has no attribute 'figure'